In [5]:
%%writefile HPC_1B.cpp
#include <iostream>
#include <vector>
#include <stack>
#include <omp.h>
using namespace std;

const int MAX = 100;

vector<int> graph[MAX];
bool visited[MAX];

// Parallel DFS function
void parallelDFS(int start) {
    stack<int> s;
    s.push(start);

    while (!s.empty()) {
        int curr;

        // Critical section for safe stack access
        #pragma omp critical
        {
            if (!s.empty()) {
                curr = s.top();
                s.pop();
            }
        }

        // Check and mark visited
        if (!visited[curr]) {
            visited[curr] = true;

            // Print node
            #pragma omp critical
            cout << curr << " ";

            // Explore neighbors in parallel
            #pragma omp parallel for
            for (int i = 0; i < graph[curr].size(); i++) {
                int adj = graph[curr][i];

                if (!visited[adj]) {
                    #pragma omp critical
                    s.push(adj);
                }
            }
        }
    }
}

int main() {
    int n, m, start;

    cout << "Enter number of nodes, edges, start node: ";
    cin >> n >> m >> start;

    cout << "Enter edges (u v):\n";
    for (int i = 0; i < m; i++) {
        int u, v;
        cin >> u >> v;

        graph[u].push_back(v);
        graph[v].push_back(u); // undirected graph
    }

    // Initialize visited array
    #pragma omp parallel for
    for (int i = 0; i < n; i++) {
        visited[i] = false;
    }

    cout << "Parallel DFS Traversal: ";
    parallelDFS(start);

    return 0;
}



Overwriting HPC_1B.cpp


In [6]:
!g++ -fopenmp HPC_1B.cpp -o HPC_1B

In [7]:
!./HPC_1B

Enter number of nodes, edges, start node: 5 5 0
Enter edges (u v):
0 1
0 2
1 3
1 4
2 4
Parallel DFS Traversal: 0 1 3 4 2 